# MLflow Model Management and Registry Tutorial

This notebook demonstrates how to:
- Register trained models in MLflow Model Registry
- Load registered models for predictions
- Manage model versions and stages
- Transition models to production

**Model Registry** is like a library for your trained models - organize, version, and deploy them easily!

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

## Step 1: Import Required Libraries

Import all necessary libraries for machine learning and MLflow tracking

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

## Step 2: Create Imbalanced Dataset

Generate synthetic data for binary classification with class imbalance (90% vs 10%)

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## Step 3: Split the Data

Divide data into training (70%) and testing (30%) sets while maintaining class balance

## Step 4: Handle Class Imbalance

Use SMOTETomek to balance the classes by creating synthetic samples of the minority class

In [5]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619], dtype=int64))

## Step 5: Define Models for Comparison

Create a list of 4 different models with their parameters and datasets

In [6]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [7]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

## Step 6: Train All Models

Train each model, make predictions, and generate performance reports

In [8]:
import mlflow

## Step 7: Import MLflow Libraries

Import MLflow modules for experiment tracking and model logging

In [12]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/01/26 09:47:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/26 09:47:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/2/runs/fe91392ac76f4c6096aa53bd5e9aaad7
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/01/26 09:47:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/2/runs/1b03371f0d494d36a18a70021626f14f
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/01/26 09:47:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/2/runs/c9c9e368d44040fc9bfd8c337e1ac6be
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/2/runs/4aeba6c4f3004c67926db3bc1d1798da
🧪 View experiment at: http://localhost:5000/#/experiments/2


## Step 8: Track Experiments in MLflow

Log all models with their parameters, metrics, and artifacts to MLflow.
Open http://localhost:5000 after running to view experiments.

## Step 9: Register the Model in MLflow Registry

**What is Model Registration?**
- Saves your trained model to a central registry
- Assigns version numbers automatically
- Makes it easy to deploy and share models

**How to use:**
1. Run your experiments in Step 8
2. Copy the Run ID from MLflow UI (http://localhost:5000)
3. Paste it in the input prompt below
4. The model will be registered with a name

In [14]:
model_name = 'XGB-Smote'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model_name'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/2/runs/c9c9e368d44040fc9bfd8c337e1ac6be
🧪 View experiment at: http://localhost:5000/#/experiments/2


Registered model 'XGB-Smote' already exists. Creating a new version of this model...


MlflowException: Unable to find a logged_model with artifact_path model_name under run c9c9e368d44040fc9bfd8c337e1ac6be

## Step 10: Load the Registered Model

**Loading a model for predictions:**
- Specify the model name and stage/version
- **@staging**: Load model from staging environment
- **@production**: Load model from production environment
- Or use version number: `/1`, `/2`, etc.

The loaded model can make predictions on new data immediately!

In [15]:
model_version = 1
model_uri = f"models:/{model_name}/@staging"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

RestException: INVALID_PARAMETER_VALUE: Invalid Model Version stage: @staging. Value must be one of None, Staging, Production, Archived.

## Step 11: Transition Model to Production

**Model Lifecycle Management:**
- Models go through stages: None → Staging → Production → Archived
- This code copies a model from staging to production
- **MlflowClient**: Provides advanced model management functions
- Use this to deploy your best model to production!

In [ ]:
current_model_uri = f"models:/{model_name}@staging"
production_model_name = "XGB-Smote"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

<ModelVersion: aliases=[], creation_timestamp=1769276157163, current_stage='None', description='', last_updated_timestamp=1769276157163, name='XGB-Smote', run_id='27c9003e205b41de8da8f437e4e54976', run_link='', source='models:/XGB-Smote/1', status='READY', status_message='', tags={}, user_id='', version='2'>

In [ ]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@staging"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

MlflowException: The following failures occurred while downloading one or more artifacts from http://localhost:5000/api/2.0/mlflow-artifacts/artifacts/2/27c9003e205b41de8da8f437e4e54976/artifacts/model_name:
##### File  #####
API request to http://localhost:5000/api/2.0/mlflow-artifacts/artifacts/2/27c9003e205b41de8da8f437e4e54976/artifacts/model_name/ failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow-artifacts/artifacts/2/27c9003e205b41de8da8f437e4e54976/artifacts/model_name/ (Caused by ResponseError('too many 500 error responses'))

## Additional Resources

To learn more about MLflow Model Registry and its advanced features:
- Visit: https://mlflow.org/docs/latest/model-registry.html
- Learn about model stages, versioning, and deployment strategies